# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
import sys
print(sys.executable)

c:\Users\ty.strong\Documents\building-agents-project\Code\project\starter\.venv\Scripts\python.exe


In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import chromadb

from dotenv import load_dotenv, find_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import (
    UserMessage,
    SystemMessage,
    ToolMessage,
    AIMessage,
)
from lib.tooling import tool

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [5]:
load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")

assert OPENAI_API_KEY, "OPENAI_API_KEY is not set"
assert TAVILY_API_KEY, "TAVILY_API_KEY is not set"
assert OPENAI_BASE_URL, "OPENAI_BASE_URL is not set"

print("Environment variables loaded")
print("OpenAI key loaded:", bool(OPENAI_API_KEY))
print("Tavily key loaded:", bool(TAVILY_API_KEY))
print("OpenAI endpoint:", OPENAI_BASE_URL)

Environment variables loaded
OpenAI key loaded: True
Tavily key loaded: True
OpenAI endpoint: https://openai.vocareum.com/v1


In [6]:
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path="chroma_data")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
    model_name="text-embedding-3-small",
)

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

print("Collection loaded:", collection.name)
print("Documents:", collection.count())

Collection loaded: udaplay
Documents: 15


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [7]:
@tool
def retrieve_game(query: str) -> list[dict]:
    """Search the local video game database for information relevant to a question."""
    results = collection.query(
        query_texts=[query],
        n_results=3,
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    return [
        {
            "document": document,
            "metadata": metadata,
            "distance": distance,
        }
        for document, metadata, distance
        in zip(documents, metadatas, distances)
    ]

#### Evaluate Retrieval Tool

In [8]:
@tool
def evaluate_retrieval(
    question: str,
    retrieved_docs: list[dict],
) -> dict:
    """Evaluate whether local search results are sufficient to answer a question."""
    prompt = f"""
You evaluate evidence for a video game research assistant.

Question:
{question}

Retrieved documents:
{json.dumps(retrieved_docs, indent=2)}

Decide whether the documents are sufficient to answer the question accurately.
Return exactly this format:

useful: true or false
description: one short explanation

Use useful=false when the documents are missing, irrelevant, outdated,
or do not contain enough evidence.
"""

    evaluator = LLM(
        model="gpt-4o-mini",
        temperature=0,
    )
    response = evaluator.invoke(prompt)

    content = response.content or ""
    useful = "useful: true" in content.lower()

    return {
        "useful": useful,
        "description": content,
    }

#### Game Web Search Tool

In [9]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def game_web_search(question: str) -> dict:
    """Search the web for video game information missing from local data."""
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
    )

    return {
        "query": question,
        "results": [
            {
                "title": result.get("title"),
                "url": result.get("url"),
                "content": result.get("content"),
            }
            for result in response.get("results", [])
        ],
    }

### Agent

In [10]:
agent = Agent(
    model_name="gpt-4o-mini",
    temperature=0,
    instructions="""
You are UdaPlay, a video game research assistant.

For every question:
1. Use retrieve_game to search the local game database.
2. Use evaluate_retrieval to determine whether the results are sufficient.
3. If the evidence is insufficient, use game_web_search.
4. Answer only with information supported by the retrieved evidence.
5. Clearly identify sources and express uncertainty when appropriate.
6. Never invent release dates, platforms, publishers, developers, or current events.
""",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
    ],
)

print("Agent created")

Agent created


In [11]:
run = agent.invoke("Who developed FIFA 21?")

final_state = run.get_final_state()

for message in final_state["messages"]:
    print(message)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
role='system' content='\nYou are UdaPlay, a video game research assistant.\n\nFor every question:\n1. Use retrieve_game to search the local game database.\n2. Use evaluate_retrieval to determine whether the results are sufficient.\n3. If the evidence is insufficient, use game_web_search.\n4. Answer only with information supported by the retrieved evidence.\n5. Clearly identify sources and express uncertainty when appropriate.\n6. Never invent release dates, platforms, publishers, developers, or current events.\n'
role='user' content='Who developed FIFA 21?

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes